# 📖 Notebook 4: API Versioning

## Why Versioning?

APIs evolve over time — you add features, fix mistakes, and improve the design.  
But **you can't just change things** because real apps and services depend on your API.  
If you change the response format, every client that parses the old format **will break**.

**API versioning** lets you release improved versions of your API while keeping the old version running  
so existing clients don't crash.

---

## 🎯 Learning Objectives

By the end of this notebook, you will understand:

1. **URL versioning** — the most common way to version an API (`/v1/...` vs `/v2/...`)
2. **Backward compatibility** — how to evolve an API without breaking existing clients
3. **Breaking vs non-breaking changes** — what's safe to change and what isn't
4. **Migration strategies** — how to roll out a new version and retire the old one

---

## 🧪 What We'll Explore

Our event ticketing API has **two versions**:

| Feature | V1 (Original) | V2 (Improved) |
|---------|---------------|---------------|
| Venue info | Just `venue_id` (a number) | Full nested `venue` object |
| Pagination | Offset-based (`offset` + `limit`) | Cursor-based (`cursor` + `limit`) |
| Status | Deprecated | Current |

We'll call both versions, compare them side by side, and learn when to create a new version.

## ⚙️ Setup

Before running this notebook, make sure the lab environment is running:

```bash
# From the api-design directory
cd 01-foundations/api-design
docker compose up -d
```

**Kernel selection:** In VS Code, click the kernel picker (top-right of the notebook)  
and select the `.venv` kernel. If it doesn't appear, reload the window  
(`Cmd+Shift+P` → "Reload Window").

**Services running:**
- FastAPI server → http://localhost:8000
- PostgreSQL → localhost:5432 (database: `api_design_demo`)
- Adminer (DB UI) → http://localhost:8080

In [ ]:
# ============================================================
# Connection Setup
# ============================================================
# We need:
#   - requests: to call our API endpoints
#   - psycopg2:  to query the database directly
#   - json:      to pretty-print JSON responses
# ============================================================

import requests
import psycopg2
import json

# --- API base URL ---
BASE_URL = "http://localhost:8000"

# --- Database connection settings ---
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "api_design_demo",
    "user": "demo",
    "password": "demo",
}


def pretty(response):
    """Pretty-print a JSON API response."""
    print(f"Status: {response.status_code}")
    print(json.dumps(response.json(), indent=2, default=str))


def query_db(sql):
    """Run a SQL query and return all rows."""
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute(sql)
    columns = [desc[0] for desc in cur.description]
    rows = cur.fetchall()
    cur.close()
    conn.close()
    return columns, rows


# --- Quick connection test ---
resp = requests.get(f"{BASE_URL}/v1/events?limit=1")
print(f"✅ API is reachable! Status: {resp.status_code}")

cols, rows = query_db("SELECT COUNT(*) FROM events")
print(f"✅ Database connected! Events in DB: {rows[0][0]}")

## 🤔 Why Version Your API?

Imagine this scenario:

> You built an events API. **1,000 mobile apps** are using it to show events to their users.  
> Every app parses the response and expects `venue_id` to be a number like `3`.  
>
> Now you want to improve the API — instead of just `venue_id: 3`, you want to include  
> the full venue details: `venue: {id: 3, name: "Madison Square Garden", city: "New York"}`.  
>
> If you just change the response format... **all 1,000 apps crash** because they're looking  
> for `venue_id` (a number) and instead they get `venue` (an object).

### 🔌 The Phone Charger Analogy

Think of it like phone chargers:

- **USB-C** is better than **micro-USB** in every way
- But you can't just **stop making micro-USB cables overnight**
- Millions of devices still use micro-USB
- So manufacturers ship **both** for a transition period
- Eventually, micro-USB is phased out

API versioning works the same way:
- **V2** is better than **V1**
- But you keep **V1 running** while clients migrate
- You set a **deprecation date** for V1
- Eventually, V1 is retired

## 🏗️ Our Two API Versions

Our event ticketing API has two versions living side by side:

### V1 — The Original Design
- Events have a `venue_id` field (just a number)
- If you need venue details, you make a **separate API call**
- Uses **offset pagination**: `?offset=0&limit=10` → skip 0, take 10
- Endpoint: `GET /v1/events`

### V2 — The Improved Design
- Events have a `venue` field with a **nested object** (id, name, city, capacity)
- One API call gives you **everything you need** — no extra requests
- Uses **cursor pagination**: `?cursor=5&limit=10` → events after ID 5
- Endpoint: `GET /v2/events`

**V2 is better**, but **V1 still works** for old clients that haven't migrated yet.  
Let's see both in action!

In [ ]:
# ============================================================
# 📦 V1 in Action — The Original API
# ============================================================
# V1 uses:
#   - Flat structure: venue_id is just a number
#   - Offset pagination: offset=0, limit=3
# ============================================================

print("=" * 60)
print("V1: List Events (offset pagination)")
print("GET /v1/events?limit=3")
print("=" * 60)

# Fetch 3 events from V1
resp = requests.get(f"{BASE_URL}/v1/events", params={"limit": 3})
data = resp.json()

# Show the full response
pretty(resp)

print("\n" + "-" * 60)
print("👆 Notice:")
print('  - Each event has "venue_id": <number>  (just an ID, no details)')
print('  - Pagination uses "offset", "limit", and "total"')

In [ ]:
# Fetch a single event from V1
print("=" * 60)
print("V1: Get Single Event")
print("GET /v1/events/1")
print("=" * 60)

resp = requests.get(f"{BASE_URL}/v1/events/1")
pretty(resp)

print("\n" + "-" * 60)
print('👆 The event has "venue_id": <number>')
print("   To get venue details, you'd need a separate call to /v1/venues/<id>")

In [ ]:
# ============================================================
# 🚀 V2 in Action — The Improved API
# ============================================================
# V2 uses:
#   - Nested structure: venue is a full object with name, city, etc.
#   - Cursor pagination: cursor=<last_id>, limit=3
# ============================================================

print("=" * 60)
print("V2: List Events (cursor pagination)")
print("GET /v2/events?limit=3")
print("=" * 60)

# Fetch 3 events from V2
resp = requests.get(f"{BASE_URL}/v2/events", params={"limit": 3})
data = resp.json()

# Show the full response
pretty(resp)

print("\n" + "-" * 60)
print("👆 Notice the improvements:")
print('  - Each event has "venue": {id, name, city, capacity}  (full details!)')
print('  - Pagination uses "next_cursor", "limit", and "has_more"')

In [ ]:
# Fetch a single event from V2
print("=" * 60)
print("V2: Get Single Event")
print("GET /v2/events/1")
print("=" * 60)

resp = requests.get(f"{BASE_URL}/v2/events/1")
pretty(resp)

print("\n" + "-" * 60)
print('👆 The venue is a full object — no extra API call needed!')
print('   One request gives you everything: event details + venue details')

In [ ]:
# ============================================================
# 🔍 Side-by-Side Comparison
# ============================================================
# Let's fetch the SAME event from both versions and compare.
# This makes the differences crystal clear.
# ============================================================

event_id = 1

# Fetch the same event from both versions
v1_resp = requests.get(f"{BASE_URL}/v1/events/{event_id}")
v2_resp = requests.get(f"{BASE_URL}/v2/events/{event_id}")

v1_event = v1_resp.json()
v2_event = v2_resp.json()

print("=" * 60)
print(f"Same event (ID={event_id}) from two API versions")
print("=" * 60)

print("\n--- V1 Response ---")
print(json.dumps(v1_event, indent=2, default=str))

print("\n--- V2 Response ---")
print(json.dumps(v2_event, indent=2, default=str))

# Highlight the key difference
print("\n" + "=" * 60)
print("🔑 Key Difference: Venue Information")
print("=" * 60)
print(f'  V1 → "venue_id": {v1_event.get("venue_id")}  (just a number)')
print(f'  V2 → "venue":    {json.dumps(v2_event.get("venue"))}  (full object!)')

In [ ]:
# ============================================================
# 🔍 Pagination Comparison
# ============================================================
# Let's also compare how pagination works in each version.
# ============================================================

v1_list = requests.get(f"{BASE_URL}/v1/events", params={"limit": 2}).json()
v2_list = requests.get(f"{BASE_URL}/v2/events", params={"limit": 2}).json()

print("=" * 60)
print("Pagination Comparison")
print("=" * 60)

print("\n--- V1 Pagination (offset-based) ---")
print(json.dumps(v1_list["pagination"], indent=2))
print("→ To get next page: ?offset=2&limit=2")

print("\n--- V2 Pagination (cursor-based) ---")
print(json.dumps(v2_list["pagination"], indent=2))
cursor = v2_list["pagination"].get("next_cursor")
print(f"→ To get next page: ?cursor={cursor}&limit=2")

print("\n" + "-" * 60)
print("Why cursor pagination is better:")
print("  - Offset: breaks when items are inserted/deleted during paging")
print("  - Cursor: always picks up where you left off, even if data changes")

## ⚠️ Breaking vs Non-Breaking Changes

When you update an API, some changes are **safe** (non-breaking) and some are **dangerous** (breaking).

### ✅ Non-Breaking Changes (Safe — No New Version Needed)

These changes **won't crash** existing clients:

| Change | Why It's Safe |
|--------|---------------|
| **Adding** a new field to the response | Old clients just ignore fields they don't know |
| **Adding** a new endpoint | Old clients don't call it, so nothing breaks |
| **Adding** an optional query parameter | Old clients don't send it, so they get default behavior |
| **Widening** a field type (int → string) | Old clients can still parse it |

### ❌ Breaking Changes (Dangerous — Requires a New Version)

These changes **will crash** existing clients:

| Change | Why It Breaks |
|--------|---------------|
| **Removing** a field | Clients that read that field will get an error |
| **Renaming** a field | Same as removing — `venue_id` → `venue` breaks parsers |
| **Changing** the response structure | Clients expect a specific shape |
| **Changing** parameter behavior | `?offset=` behaving like `?cursor=` confuses clients |
| **Changing** a field type | `venue_id: 3` (number) → `venue: {...}` (object) crashes JSON parsing |

---

**Rule of thumb:**  
If a client written for the old version would **crash or get wrong data** with the new version,  
it's a **breaking change** and you need a **new version**.

In [ ]:
# ============================================================
# 💥 Demonstration: Why V1 → V2 is a Breaking Change
# ============================================================
# Pretend we're a V1 client that expects `venue_id` to be a plain integer.
# What happens when the same client accidentally calls V2?
# ============================================================

def v1_client_parse_event(event_data):
    """A client written for V1. Expects `venue_id` to be a simple integer."""
    title = event_data["title"]
    venue_id = event_data["venue_id"]   # V1 clients use this as a number
    return f"Event: {title}, Venue ID: {venue_id}"


# NOTE: both /v1/events/1 and /v2/events/1 wrap the row in {"event": {...}}.
# A real V1 client would unwrap it the same way.
v1_body = requests.get(f"{BASE_URL}/v1/events/1").json()
v2_body = requests.get(f"{BASE_URL}/v2/events/1").json()

print("✅ V1 client + V1 API:")
print(f"   {v1_client_parse_event(v1_body['event'])}")
print()

print("❌ V1 client + V2 API:")
try:
    print(f"   {v1_client_parse_event(v2_body['event'])}")
except KeyError as e:
    print(f"   💥 CRASH! KeyError: {e}")
    print(f"   The client expected 'venue_id' but V2 has 'venue' (an object).")
    print(f"   V2 venue field: {json.dumps(v2_body['event'].get('venue'))}")

print()
print("This is exactly why we need versioning!")
print("V1 clients keep using /v1/events and everything works.")
print("New clients use /v2/events and get the improved format.")


## 🗂️ Versioning Strategies

There are several ways to version an API. Here are the three most common:

---

### 1. URL Versioning (Our Approach — Most Common)

```
GET /v1/events
GET /v2/events
```

| Pros | Cons |
|------|------|
| ✅ Explicit — you can see the version in the URL | ❌ URL changes between versions |
| ✅ Easy to understand — even beginners get it | ❌ Some code duplication on the server |
| ✅ Easy to route — load balancers can direct traffic | |
| ✅ Easy to test — just change the URL in your browser | |

**Used by:** Twitter, Stripe, GitHub, Google Maps

---

### 2. Header Versioning

```
GET /events
Accept-Version: v2
```

| Pros | Cons |
|------|------|
| ✅ Clean URLs — the URL doesn't change | ❌ Harder to test (you need to set headers) |
| ✅ Follows HTTP semantics | ❌ Less discoverable — can't see version in URL |

**Used by:** GitHub (also supports this), Azure

---

### 3. Query Parameter Versioning

```
GET /events?version=2
```

| Pros | Cons |
|------|------|
| ✅ Simple to add | ❌ Easy to forget the parameter |
| ✅ Works with any HTTP client | ❌ What happens if you don't specify a version? |

---

### 🎤 In Interviews

**URL versioning is the safe default.** If asked about API versioning in a system design interview,  
say: *"I'd use URL versioning like `/v1/events` — it's the most common approach,  
it's explicit, easy to route, and easy to understand."*

## 🚀 Migration Strategy

When you release a new API version, you don't just flip a switch.  
Here's the typical process:

### Step-by-Step Migration

1. **Deploy V2 alongside V1** — both versions run at the same time
2. **Document the changes** — publish a migration guide explaining what changed
3. **Notify developers** — email, blog post, changelog: *"V2 is available!"*
4. **Set a deprecation timeline** — *"V1 will be retired on January 1, 2025"*
5. **Add deprecation warnings** — V1 responses can include a header: `Deprecated: true`
6. **Monitor V1 usage** — track how many clients still use V1
7. **Sunset V1** — once usage is low enough, shut it down

Let's check our database to see how we track API versions.

### 🪦 Standard Deprecation Headers in the Response

Instead of only documenting "V1 is deprecated" on a web page, our server
attaches two **standard HTTP headers** to every V1 response:

| Header        | Meaning                                                              |
|---------------|----------------------------------------------------------------------|
| `Deprecation` | `true` (or a date) — "this endpoint is deprecated"                   |
| `Sunset`      | HTTP date — "this endpoint will be shut down on this day"            |
| `Link`        | points to the successor resource (V2) with `rel="successor-version"` |

Well-behaved clients can detect these and surface a warning in their logs,
long before the removal date arrives.


In [ ]:
# Same request to V1 and V2 — compare the response headers.
v1 = requests.get(f"{BASE_URL}/v1/events?limit=1")
v2 = requests.get(f"{BASE_URL}/v2/events?limit=1")

def show(name, resp):
    print(f"--- {name} ({resp.status_code}) ---")
    for h in ("Deprecation", "Sunset", "Link"):
        print(f"  {h:<12} = {resp.headers.get(h, '(not set)')}")

show("V1", v1)
print()
show("V2", v2)

print()
print("💡 A production client would log:")
print("   ⚠️  'GET /v1/events is deprecated; plan to migrate before <Sunset date>'.")


In [ ]:
# ============================================================
# 📊 Check API Version Status in the Database
# ============================================================
# Our database has an api_versions table that tracks which
# versions are active and which are deprecated.
# ============================================================

columns, rows = query_db("SELECT * FROM api_versions ORDER BY version")

print("=" * 60)
print("API Versions (from database)")
print("=" * 60)

# Print as a nice table
header = " | ".join(f"{col:>12}" for col in columns)
print(header)
print("-" * len(header))

for row in rows:
    values = " | ".join(f"{str(val):>12}" for val in row)
    print(values)

print()
print("📌 Key observations:")
print("  - V1 is marked as DEPRECATED (deprecated = True)")
print("  - V2 is the CURRENT version (deprecated = False)")
print("  - V1 still works — it's deprecated, not removed")
print("  - Clients should migrate to V2 before V1 is shut down")

In [ ]:
# ============================================================
# 🤖 Building a Version-Aware Client
# ============================================================
# A well-designed client can work with multiple API versions.
# This makes migration easier — just change the version number
# and the client adapts automatically.
# ============================================================


class EventClient:
    """
    A simple client that works with both V1 and V2 of our events API.

    Usage:
        client = EventClient(api_version="v2")
        events = client.list_events(limit=5)
        event  = client.get_event(1)
    """

    def __init__(self, api_version="v1", base_url="http://localhost:8000"):
        # Store the version — this determines which endpoints we call
        self.api_version = api_version
        self.base_url = f"{base_url}/{api_version}"
        print(f"EventClient initialized with API {api_version}")

    def list_events(self, limit=10, **kwargs):
        """
        Fetch a list of events.
        V1 uses offset pagination, V2 uses cursor pagination.
        """
        params = {"limit": limit}

        if self.api_version == "v1":
            # V1: offset pagination
            params["offset"] = kwargs.get("offset", 0)
        elif self.api_version == "v2":
            # V2: cursor pagination
            cursor = kwargs.get("cursor")
            if cursor is not None:
                params["cursor"] = cursor

        resp = requests.get(f"{self.base_url}/events", params=params)
        return resp.json()

    def get_event(self, event_id):
        """Fetch a single event by ID and unwrap the envelope.

        Both versions wrap the row in ``{"event": {...}}``; we unwrap so that
        callers get the event itself regardless of version.
        """
        resp = requests.get(f"{self.base_url}/events/{event_id}")
        return resp.json()["event"]

    def get_venue_name(self, event):
        """
        Get the venue name from an event, handling version differences.
        V1: venue_id is a number — we'd need an extra API call.
        V2: venue is a nested object — name is right there.
        """
        if self.api_version == "v2":
            # V2 has the venue nested right in the event
            return event.get("venue", {}).get("name", "Unknown")
        else:
            # V1 only has venue_id — we can't get the name without another call
            return f"(venue_id={event.get('venue_id')} — name not available in V1)"


# ============================================================
# Test with V1
# ============================================================
print("=" * 60)
v1_client = EventClient(api_version="v1")
print("=" * 60)

event = v1_client.get_event(1)
print(f"Event: {event['title']}")
print(f"Venue: {v1_client.get_venue_name(event)}")

# ============================================================
# Test with V2
# ============================================================
print()
print("=" * 60)
v2_client = EventClient(api_version="v2")
print("=" * 60)

event = v2_client.get_event(1)
print(f"Event: {event['title']}")
print(f"Venue: {v2_client.get_venue_name(event)}")

print()
print("💡 Same client class, same method calls — but V2 gives us richer data!")

In [ ]:
# ============================================================
# Show the client working with pagination in both versions
# ============================================================

print("=" * 60)
print("Pagination with V1 client (offset-based)")
print("=" * 60)

# V1: page through events using offset
page1 = v1_client.list_events(limit=2, offset=0)
print(f"Page 1 events: {[e['title'] for e in page1['events']]}")
print(f"Pagination: {page1['pagination']}")

page2 = v1_client.list_events(limit=2, offset=2)
print(f"Page 2 events: {[e['title'] for e in page2['events']]}")
print(f"Pagination: {page2['pagination']}")

print()
print("=" * 60)
print("Pagination with V2 client (cursor-based)")
print("=" * 60)

# V2: page through events using cursor
page1 = v2_client.list_events(limit=2)
print(f"Page 1 events: {[e['title'] for e in page1['events']]}")
print(f"Pagination: {page1['pagination']}")

# Use the next_cursor from page 1 to get page 2
next_cursor = page1["pagination"]["next_cursor"]
page2 = v2_client.list_events(limit=2, cursor=next_cursor)
print(f"Page 2 events: {[e['title'] for e in page2['events']]}")
print(f"Pagination: {page2['pagination']}")

## 📝 Key Takeaways

### 1. Version Your API From Day One
Always start with `/v1/...` — even if you don't plan a V2 yet.  
It costs nothing to add and saves a huge headache later.

### 2. Non-Breaking Changes Are Free
You can **add new fields** to your response anytime without a new version.  
Well-written clients ignore fields they don't recognize.

### 3. Breaking Changes Need a New Version
If you **remove a field**, **rename a field**, or **change the structure**,  
you need a new version (`/v2/...`). Never break existing clients.

### 4. URL Versioning is the Simplest
`/v1/events` vs `/v2/events` — explicit, easy to understand, easy to route.  
This is the most widely used approach (Twitter, Stripe, GitHub, Google).

### 5. Keep Old Versions Running During Migration
Don't just switch off V1. Give clients time to migrate:  
Deploy V2 → Document changes → Set deprecation date → Sunset V1.

### 6. Interview Tip 🎤
Mentioning API versioning in a system design interview shows **production awareness**.  
It tells the interviewer you think about real-world concerns like backward compatibility,  
client migration, and the evolution of APIs over time.

---

**What's next?** Try modifying the `EventClient` class to automatically detect  
the API version and adapt. Or explore the other notebooks in this lab series!